In [ ]:
# @title Model G - ResNet50 scratch
# ============================================================
# MODEL G
# ResNet50 from scratch - 224x224
# WITH augmentation
# FULL STATE CHECKPOINTING (local )
# ============================================================

# ------------------------------------------------------------
# Clean memory before starting
# ------------------------------------------------------------
import gc
import os
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


LOCAL_RUN_NAME = "model_i10_efficientNetB0_noaug_partial"


BASE_DIR = Path.cwd().parent.parent.parent
LOCAL_DATA_DIR = BASE_DIR / "data"

OUTPUT_DIR_BASE = BASE_DIR / "outputs" / "image_modeling" / LOCAL_RUN_NAME
MODEL_DIR_BASE = BASE_DIR / "data" / "models" / LOCAL_RUN_NAME
FIGURE_DIR_BASE = BASE_DIR / "figures" / LOCAL_RUN_NAME

OUTPUT_DIR_BASE.mkdir(parents=True, exist_ok=True)
MODEL_DIR_BASE.mkdir(parents=True, exist_ok=True)
FIGURE_DIR_BASE.mkdir(parents=True, exist_ok=True)

DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
train_df = pd.read_csv(SPLIT_DIR / "train_split.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json") as f:
    label2id = {int(k): int(v) for k, v in json.load(f).items()}


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Run folders
# ------------------------------------------------------------
# LOCAL_RUN_NAME = "resnet50_scratch_224_aug"

LOCAL_RUN_NAME = "model_i8_restNet50_moderate_aug_from_scratch"
LOCAL_OUTPUT_ROOT = LOCAL_DATA_DIR / "outputs" / LOCAL_RUN_NAME
LOCAL_MODEL_ROOT = LOCAL_DATA_DIR / "models" / LOCAL_RUN_NAME

for d in [
    LOCAL_OUTPUT_ROOT, LOCAL_MODEL_ROOT,
]:
    d.mkdir(parents=True, exist_ok=True)

print("Local output root:", LOCAL_OUTPUT_ROOT)

# ------------------------------------------------------------
# Parameters
# ------------------------------------------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
MODEL_NAME = "ResNet50_Scratch_224_Aug"
MAX_EPOCHS = 18
EARLY_STOPPING_PATIENCE = 5

RESUME_TRAINING = False
CHECKPOINT_SOURCE = "local_last"   # local_last, local_best

# ------------------------------------------------------------
# Transforms
# Same augmentation pattern as Model C
# ------------------------------------------------------------
train_transform_G = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 20, IMAGE_SIZE + 20)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform_G = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
class RakutenImageDataset(Dataset):
    def __init__(self, df, transform=None, path_col="image_path_local", label_col="label_id"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.path_col = path_col
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, self.path_col]
        label = int(self.df.loc[idx, self.label_col])

        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return image, label

train_dataset_G = RakutenImageDataset(train_df, transform=train_transform_G)
val_dataset_G = RakutenImageDataset(val_df, transform=val_transform_G)

train_loader_G = DataLoader(
    train_dataset_G,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader_G = DataLoader(
    val_dataset_G,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# ------------------------------------------------------------
# Model: ResNet50 from scratch
# ------------------------------------------------------------
model_G = models.resnet50(weights=None)
in_features = model_G.fc.in_features
model_G.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, len(label2id))
)

model_G = model_G.to(device)
print(model_G)

trainable_params = sum(p.numel() for p in model_G.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model_G.parameters())
print(f"Trainable params: {trainable_params:,}")
print(f"Total params    : {all_params:,}")

# ------------------------------------------------------------
# Loss, optimizer, scheduler
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model_G.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------
LAST_CKPT_LOCAL = LOCAL_MODEL_ROOT / "last_checkpoint.pt"

BEST_CKPT_LOCAL = LOCAL_MODEL_ROOT / "best_checkpoint.pt"

BEST_WEIGHTS_LOCAL = LOCAL_MODEL_ROOT / "best_model_state_dict.pt"

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def copy_to_local(local_path):
    shutil.copy2(local_path)

def save_full_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    history,
    best_macro_f1,
    best_epoch,
    model_name,
    y_true=None,
    y_pred=None,
    y_proba=None
):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
        "best_macro_f1": best_macro_f1,
        "best_epoch": best_epoch,
        "model_name": model_name,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
        "y_true_last_val": y_true,
        "y_pred_last_val": y_pred,
        "y_proba_last_val": y_proba,
    }

    if torch.cuda.is_available():
        checkpoint["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    torch.save(checkpoint, checkpoint_path)

def load_full_checkpoint(checkpoint_path, model, optimizer, scheduler, device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if "torch_rng_state" in checkpoint:
        torch.set_rng_state(checkpoint["torch_rng_state"])
    if "numpy_rng_state" in checkpoint:
        np.random.set_state(checkpoint["numpy_rng_state"])
    if "python_rng_state" in checkpoint:
        random.setstate(checkpoint["python_rng_state"])
    if torch.cuda.is_available() and "cuda_rng_state_all" in checkpoint:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])

    start_epoch = checkpoint["epoch"] + 1
    history = checkpoint.get("history", [])
    best_macro_f1 = checkpoint.get("best_macro_f1", -np.inf)
    best_epoch = checkpoint.get("best_epoch", -1)

    print(f"Resumed from checkpoint: {checkpoint_path}")
    print(f"Starting at epoch: {start_epoch}")
    print(f"Best macro F1 so far: {best_macro_f1:.4f} at epoch {best_epoch}")

    return start_epoch, history, best_macro_f1, best_epoch

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_preds = []
    all_true = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_true.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds)
    epoch_macro_f1 = f1_score(all_true, all_preds, average="macro")
    epoch_weighted_f1 = f1_score(all_true, all_preds, average="weighted")

    return epoch_loss, epoch_acc, epoch_macro_f1, epoch_weighted_f1, np.array(all_true), np.array(all_preds)

def predict_with_proba(model, loader):
    model.eval()
    all_probs = []
    all_preds = []
    all_true = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_true.append(labels.cpu().numpy())

    y_proba = np.concatenate(all_probs)
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_true)

    return y_true, y_pred, y_proba

def plot_history(history_df, save_dir_local):
    # Loss
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{MODEL_NAME} - Loss")
    plt.legend()
    plt.tight_layout()
    loss_local = save_dir_local / "training_loss.png"
    plt.savefig(loss_local, dpi=200, bbox_inches="tight")
    shutil.copy2(loss_local)
    plt.show()

    # Accuracy
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="Train Accuracy")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{MODEL_NAME} - Accuracy")
    plt.legend()
    plt.tight_layout()
    acc_local = save_dir_local / "training_accuracy.png"
    plt.savefig(acc_local, dpi=200, bbox_inches="tight")
    shutil.copy2(acc_local)
    plt.show()

    # Macro F1
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_macro_f1"], label="Train Macro F1")
    plt.plot(history_df["epoch"], history_df["val_macro_f1"], label="Val Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.title(f"{MODEL_NAME} - Macro F1")
    plt.legend()
    plt.tight_layout()
    f1_local = save_dir_local / "training_macro_f1.png"
    plt.savefig(f1_local, dpi=200, bbox_inches="tight")
    shutil.copy2(f1_local)
    plt.show()

def save_history_and_metadata(history, best_epoch, best_macro_f1, total_minutes):
    history_df = pd.DataFrame(history)
    history_local = LOCAL_OUTPUT_ROOT / "training_history.csv"
    history_df.to_csv(history_local, index=False)
    copy_to_local(history_local)

    metadata = {
        "model_name": MODEL_NAME,
        "image_size": IMAGE_SIZE,
        "augmentation": True,
        "architecture": "ResNet50_Scratch",
        "unfrozen_blocks": "all",
        "epochs_trained": int(len(history)),
        "best_epoch": int(best_epoch),
        "batch_size": int(BATCH_SIZE),
        "optimizer": "AdamW",
        "initial_lr": 3e-4,
        "weight_decay": 1e-4,
        "scheduler": "ReduceLROnPlateau",
        "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
        "best_macro_f1": float(best_macro_f1),
        "training_minutes": float(total_minutes),
        "trainable_parameters": int(trainable_params),
        "total_parameters": int(all_params)
    }

    meta_local = LOCAL_OUTPUT_ROOT / "run_metadata.json"
    with open(meta_local, "w") as f:
        json.dump(metadata, f, indent=2)
    copy_to_local(meta_local)

# ------------------------------------------------------------
# Resume option
# ------------------------------------------------------------
start_epoch = 1
history = []
best_macro_f1 = -np.inf
best_epoch = -1
epochs_without_improvement = 0

if RESUME_TRAINING:
    checkpoint_map = {
        "local_last": LAST_CKPT_LOCAL,
        "local_best": BEST_CKPT_LOCAL,
    }
    checkpoint_path = checkpoint_map[CHECKPOINT_SOURCE]

    if checkpoint_path.exists():
        start_epoch, history, best_macro_f1, best_epoch = load_full_checkpoint(
            checkpoint_path,
            model_G,
            optimizer,
            scheduler,
            device
        )
        epochs_without_improvement = max(0, start_epoch - best_epoch - 1)
    else:
        print(f"Resume requested but checkpoint not found: {checkpoint_path}")
        print("Starting from scratch instead.")

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------
start_time = time.time()

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    train_loss, train_acc, train_macro_f1, train_weighted_f1, _, _ = run_epoch(
        model_G, train_loader_G, criterion, optimizer=optimizer
    )

    val_loss, val_acc, val_macro_f1, val_weighted_f1, _, _ = run_epoch(
        model_G, val_loader_G, criterion, optimizer=None
    )

    y_true_val, y_pred_val, y_proba_val = predict_with_proba(model_G, val_loader_G)

    scheduler.step(val_macro_f1)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "train_macro_f1": train_macro_f1,
        "train_weighted_f1": train_weighted_f1,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "val_macro_f1": val_macro_f1,
        "val_weighted_f1": val_weighted_f1,
        "lr": current_lr
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f} | "
        f"train_macro_f1={train_macro_f1:.4f} | val_macro_f1={val_macro_f1:.4f} | "
        f"lr={current_lr:.6f}"
    )

    # Save last checkpoint every epoch
    save_full_checkpoint(
        checkpoint_path=LAST_CKPT_LOCAL,
        epoch=epoch,
        model=model_G,
        optimizer=optimizer,
        scheduler=scheduler,
        history=history,
        best_macro_f1=best_macro_f1,
        best_epoch=best_epoch,
        model_name=MODEL_NAME,
        y_true=y_true_val,
        y_pred=y_pred_val,
        y_proba=y_proba_val
    )
    copy_to_local(LAST_CKPT_LOCAL)

    # Save best checkpoint if improved
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_epoch = epoch
        epochs_without_improvement = 0

        save_full_checkpoint(
            checkpoint_path=BEST_CKPT_LOCAL,
            epoch=epoch,
            model=model_G,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            best_macro_f1=best_macro_f1,
            best_epoch=best_epoch,
            model_name=MODEL_NAME,
            y_true=y_true_val,
            y_pred=y_pred_val,
            y_proba=y_proba_val
        )
        copy_to_local(BEST_CKPT_LOCAL)

        torch.save(model_G.state_dict(), BEST_WEIGHTS_LOCAL)
        copy_to_local(BEST_WEIGHTS_LOCAL)

        print(f"  -> New best model saved at epoch {epoch} (val_macro_f1={val_macro_f1:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement for {epochs_without_improvement} epoch(s)")

    elapsed_minutes = (time.time() - start_time) / 60
    save_history_and_metadata(history, best_epoch, best_macro_f1, elapsed_minutes)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered after epoch {epoch}")
        break

total_time = time.time() - start_time
print(f"\nTraining finished in {total_time/60:.2f} minutes")
print(f"Best epoch: {best_epoch}")
print(f"Best val macro F1: {best_macro_f1:.6f}")

# ------------------------------------------------------------
# Load best model weights
# ------------------------------------------------------------
model_G.load_state_dict(torch.load(BEST_WEIGHTS_LOCAL, map_location=device))

# ------------------------------------------------------------
# Final validation predictions
# ------------------------------------------------------------
y_true, y_pred, y_proba = predict_with_proba(model_G, val_loader_G)

# ------------------------------------------------------------
# Final metrics
# ------------------------------------------------------------
final_accuracy = accuracy_score(y_true, y_pred)
final_macro_f1 = f1_score(y_true, y_pred, average="macro")
final_weighted_f1 = f1_score(y_true, y_pred, average="weighted")

metrics_df = pd.DataFrame([{
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "augmentation": True,
    "architecture": "ResNet50_Scratch",
    "unfrozen_blocks": "all",
    "epochs_trained": len(history),
    "best_epoch": best_epoch,
    "batch_size": BATCH_SIZE,
    "optimizer": "AdamW",
    "initial_lr": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "accuracy": final_accuracy,
    "macro_f1": final_macro_f1,
    "weighted_f1": final_weighted_f1,
    "training_minutes": total_time / 60
}])

print("\nFinal validation metrics:")
print(metrics_df)

# ------------------------------------------------------------
# Save metrics/history
# ------------------------------------------------------------
history_df = pd.DataFrame(history)

metrics_local = LOCAL_OUTPUT_ROOT / "metrics.csv"
history_local = LOCAL_OUTPUT_ROOT / "training_history.csv"

metrics_df.to_csv(metrics_local, index=False)
history_df.to_csv(history_local, index=False)

# ------------------------------------------------------------
# Save predictions/probabilities
# ------------------------------------------------------------
pred_df = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred
})

pred_local = LOCAL_OUTPUT_ROOT / "val_predictions.csv"
pred_df.to_csv(pred_local, index=False)
shutil.copy2(pred_local)

proba_local = LOCAL_OUTPUT_ROOT / "y_proba.npy"
np.save(proba_local, y_proba)
shutil.copy2(proba_local)

# ------------------------------------------------------------
# Save classification report
# ------------------------------------------------------------
report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose().reset_index().rename(columns={"index": "class_or_avg"})

report_local = LOCAL_OUTPUT_ROOT / "classification_report.csv"
report_df.to_csv(report_local, index=False)
shutil.copy2(report_local)

# ------------------------------------------------------------
# Save confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation="nearest")
plt.title(f"{MODEL_NAME} - Confusion Matrix")
plt.colorbar()
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.tight_layout()

cm_local = LOCAL_OUTPUT_ROOT / "confusion_matrix.png"
plt.savefig(cm_local, dpi=200, bbox_inches="tight")
shutil.copy2(cm_local)
plt.show()

# ------------------------------------------------------------
# Save training curves
# ------------------------------------------------------------
plot_history(history_df, LOCAL_OUTPUT_ROOT)

# ------------------------------------------------------------
# Save final metadata
# ------------------------------------------------------------
metadata = {
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "augmentation": True,
    "architecture": "ResNet50_Scratch",
    "unfrozen_blocks": "all",
    "epochs_trained": int(len(history)),
    "best_epoch": int(best_epoch),
    "batch_size": int(BATCH_SIZE),
    "optimizer": "AdamW",
    "initial_lr": 3e-4,
    "weight_decay": 1e-4,
    "scheduler": "ReduceLROnPlateau",
    "early_stopping_patience": int(EARLY_STOPPING_PATIENCE),
    "accuracy": float(final_accuracy),
    "macro_f1": float(final_macro_f1),
    "weighted_f1": float(final_weighted_f1),
    "training_minutes": float(total_time / 60),
    "trainable_parameters": int(trainable_params),
    "total_parameters": int(all_params)
}

meta_local = LOCAL_OUTPUT_ROOT / "run_metadata.json"
with open(meta_local, "w") as f:
    json.dump(metadata, f, indent=2)
shutil.copy2(meta_local)

print("\nSaved files:")
print("Last checkpoint local :", LAST_CKPT_LOCAL)
print("Best checkpoint local :", BEST_CKPT_LOCAL)
print("Best weights local    :", BEST_WEIGHTS_LOCAL)
print("Metrics local         :", metrics_local)
print("History local         :", history_local)
print("Predictions           :", pred_local)
print("Probabilities         :", proba_local)
print("Class report          :", report_local)
print("Conf matrix fig       :", cm_local)
print("Curves dir            :", LOCAL_OUTPUT_ROOT)
print("Metadata              :", meta_local)